In [1]:
import pandas as pd
import numpy as np
import re
import string

# Impor NLTK
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC  # <-- DIGANTI: Impor LinearSVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [2]:
# Ambil stopwords bahasa Inggris
stop_words = set(stopwords.words('english'))
# Inisialisasi lemmatizer
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Fungsi untuk membersihkan dan memproses teks."""
    if not isinstance(text, str):
        return ""
    
    # 1. Ubah ke huruf kecil
    text = text.lower()
    
    # 2. Hapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Hapus angka
    text = re.sub(r'\d+', '', text)
    
    # 4. Hapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 5. Tokenisasi (di notebook Anda, ini implisit, di sini kita buat eksplisit)
    tokens = text.split()
    
    # 6. Hapus stopwords dan lakukan lemmatization
    clean_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    # 7. Gabungkan kembali menjadi string
    return " ".join(clean_tokens)

In [3]:
try:
    df_train = pd.read_csv("./twitter_training.csv", header=None)
    df_val = pd.read_csv("./twitter_validation.csv", header=None)
except FileNotFoundError:
    print("Pastikan file 'twitter_training.csv' dan 'twitter_validation.csv' berada di direktori yang sama.")
    # Keluar dari skrip jika file tidak ditemukan
    # exit()

# Beri nama kolom SEBELUM di-drop
df_train.columns = ['ID', 'Borderlands', 'sentiment', 'text']
df_val.columns = ['ID', 'Borderlands', 'sentiment', 'text']

# Hapus kolom yang tidak terpakai
df_train = df_train.drop(columns=['ID', 'Borderlands'])
df_val = df_val.drop(columns=['ID', 'Borderlands'])

# Membersihkan data yang hilang (missing values)
df_train = df_train.dropna(subset=['text'])
df_val = df_val.dropna(subset=['text'])

# Membersihkan data duplikat
df_train = df_train.drop_duplicates()
df_val = df_val.drop_duplicates()

In [6]:
print("Memulai preprocessing teks data latih...")
df_train['clean_text'] = df_train['text'].apply(preprocess_text)

print("Memulai preprocessing teks data validasi...")
df_val['clean_text'] = df_val['text'].apply(preprocess_text)

print("oke selesai")

Memulai preprocessing teks data latih...
Memulai preprocessing teks data validasi...
oke selesai


In [7]:
vectorizer = TfidfVectorizer(max_features=5000)
label_encoder = LabelEncoder()

all_sentiments = pd.concat([df_train['sentiment'], df_val['sentiment']])
label_encoder.fit(all_sentiments)

y_train = label_encoder.transform(df_train['sentiment'])
y_test = label_encoder.transform(df_val['sentiment'])

X_train = vectorizer.fit_transform(df_train['clean_text'])
X_test = vectorizer.transform(df_val['clean_text'])

print(f"Bentuk X_train: {X_train.shape}")
print(f"Bentuk X_test: {X_test.shape}")

Bentuk X_train: (69769, 5000)
Bentuk X_test: (999, 5000)


In [10]:
print("\nMulai melatih model SVM (LinearSVC)...")
# SVM (LinearSVC) biasanya sangat cepat untuk data teks.

svm_model = LinearSVC(random_state=42)

# Latih model
svm_model.fit(X_train, y_train)

print("Pelatihan SVM selesai")


Mulai melatih model SVM (LinearSVC)...
Pelatihan SVM selesai


In [11]:
y_pred_svm = svm_model.predict(X_test)

y_pred_labels = label_encoder.inverse_transform(y_pred_svm)
y_test_labels = label_encoder.inverse_transform(y_test)

accuracy_svm = accuracy_score(y_test_labels, y_pred_labels)
print(f"\nAkurasi SVM (LinearSVC): {accuracy_svm * 100:.2f}%")

# Tampilkan classification report
print("\nClassification Report untuk SVM (LinearSVC):")
# Tentukan 'labels' untuk memastikan urutan yang benar
target_names = label_encoder.classes_
print(classification_report(y_test_labels, y_pred_labels, labels=target_names))


Akurasi SVM (LinearSVC): 83.08%

Classification Report untuk SVM (LinearSVC):
              precision    recall  f1-score   support

  Irrelevant       0.83      0.75      0.79       172
    Negative       0.80      0.90      0.85       266
     Neutral       0.88      0.79      0.83       285
    Positive       0.82      0.86      0.84       276

    accuracy                           0.83       999
   macro avg       0.83      0.82      0.83       999
weighted avg       0.83      0.83      0.83       999

